# Tutorial: Using the `MILDataset` and PyTorch `DataLoader` in `cellink`

This tutorial shows how to use the `MILDataset` together with PyTorch's `DataLoader` to prepare multimodal input for model training in the `cellink` framework. This pipeline is designed to support **donor-level learning** using single-cell transcriptomic measurements.

We will use the `OneK1K` dataset and demonstrate how to:

- Filter for CD8 Naive T cells,
- Package the data using `DonorData`,
- Wrap it into a `MILDataset`,
- Load it with a PyTorch `DataLoader`,
- Train a `DonorMILModel`.


## Setup and Configuration

We start by importing relevant modules and creating local directories to store input/output files. This ensures that any annotation tools have a consistent file structure to work with.

In [1]:
from cellink.resources import get_dummy_onek1k
from cellink.ml import DonorMILModel, MILDataset, mil_collate_fn
from torch.utils.data import DataLoader
import pytorch_lightning as pl
import numpy as np

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Load Genotype Data (`gdata`)

We load the example dataset using `get_dummy_onek1k()` (This is a subset of the full OneK1K dataset, which can be downloaded, and prepared using `get_onek1k()`). The `DonorData` object contains a `.D` attribute (`gdata`) that stores genotype information at the variant level. These variants will be the target of our annotations. We filter for chromosome 22 and CD8 Naive for fast execution of the notebook.

In [2]:
dd = get_dummy_onek1k()
dd

[2026-11-14 19:19:43,950] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.dd.h5 already exists
[2026-11-14 19:19:43,951] INFO:root: Verifying checksum
[2026-11-14 19:19:48,791] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)


╔═ DonorData(n_donors=300, n_cells_per_donor=[333-420], donor_id='donor_id') ═════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 300 × 146,939 │ AnnData object with n_obs × n_vars = 125,913 ×     │ ║
║ │                                                    │ 34,830                                             │ ║
║ │     var: 'AF', 'AN', 'ER2', 'IMPUTED', 'maf',      │     obs: 'orig.ident', 'nCount_RNA',               │ ║
║ │ 'NS', 'R2', 'TYPED', 'TYPED_ONLY', 'id',           │ 'nFeature_RNA', 'percent.mt', 'donor_id',          │ ║
║ │ 'id_mask', 'length', 'pos', 'quality', 'a0', 'a1', │ 'pool_number', 'predicted.celltype.l2',        

In [3]:
chrom = 22
dd = dd.sel(D_var=dd.D.var.chrom == str(chrom), C_var=dd.C.var.chrom == str(chrom)).copy()
dd

╔═ DonorData(n_donors=300, n_cells_per_donor=[333-420], donor_id='donor_id') ═════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 300 × 136,776 │ AnnData object with n_obs × n_vars = 125,913 × 888 │ ║
║ │     var: 'AF', 'AN', 'ER2', 'IMPUTED', 'maf',      │     obs: 'orig.ident', 'nCount_RNA',               │ ║
║ │ 'NS', 'R2', 'TYPED', 'TYPED_ONLY', 'id',           │ 'nFeature_RNA', 'percent.mt', 'donor_id',          │ ║
║ │ 'id_mask', 'length', 'pos', 'quality', 'a0', 'a1', │ 'pool_number', 'predicted.celltype.l2',            │ ║
║ │ 'chrom', 'pos_hg19', 'id_hg19'                     │ 'predicted.celltype.l2.score', 'age',          

In [4]:
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
dd = dd[..., dd.C.obs[celltype_key] == cell_type, :].copy()
dd

╔═ DonorData(n_donors=294, n_cells_per_donor=[1-137], donor_id='donor_id') ═══════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 294 × 136,776 │ AnnData object with n_obs × n_vars = 4,859 × 888   │ ║
║ │     var: 'AF', 'AN', 'ER2', 'IMPUTED', 'maf',      │     obs: 'orig.ident', 'nCount_RNA',               │ ║
║ │ 'NS', 'R2', 'TYPED', 'TYPED_ONLY', 'id',           │ 'nFeature_RNA', 'percent.mt', 'donor_id',          │ ║
║ │ 'id_mask', 'length', 'pos', 'quality', 'a0', 'a1', │ 'pool_number', 'predicted.celltype.l2',            │ ║
║ │ 'chrom', 'pos_hg19', 'id_hg19'                     │ 'predicted.celltype.l2.score', 'age',          

## Wrap the Data with `MILDataset`
We use the `MILDataset` to create a wrapper that returns bags of cells for each donor. This enables multiple instance learning (MIL) with donor labels. The `MILDataset` automatically packages all labels, categorical and continuous covariates and data matrices, when available. You may adjust the keys. Please note, that for demosntration purposes we are randomly generating labels now.

In [5]:
dd.D.obs["donor_labels"] = np.random.randint(2, size=len(dd.D.obs))
dd.C.obs["pool_number"] = dd.C.obs["pool_number"].astype("float")

In [6]:
dataset = MILDataset(
    dd,
    donor_labels_key="donor_labels",
    cell_batch_key="pool_number",
    # split_donors=["OneK1K_387", "OneK1K_809", "OneK1K_137"],
    split_indices=list(range(10)),
)

## Create a PyTorch `DataLoader`
We now build a PyTorch `DataLoader` that will feed batches of donors (each with a variable number of cells) into the model. We use the `mil_collate_fn` to handle the collation of variable-length inputs.

In [7]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=mil_collate_fn)

## Initialize and Train a Model
We now create a `DonorMILModel`, specifying the dimensionality of the donor and cell input features. We then use PyTorch Lightning to train for two epochs.

In [8]:
model = DonorMILModel(n_input_donor=dd.D.n_vars, n_input_cell=dd.C.n_vars)

In [9]:
trainer = pl.Trainer(max_epochs=2, accelerator="cpu")  # the model is small; MPS on macOS 14 aborts torch
trainer.fit(model, dataloader)

site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /path/to/cellink_ ...


[2026-11-14 19:19:51,228] INFO:pytorch_lightning.utilities.rank_zero: GPU available: False, used: False
[2026-11-14 19:19:51,230] INFO:pytorch_lightning.utilities.rank_zero: TPU available: False, using: 0 TPU cores
[2026-11-14 19:19:51,233] INFO:pytorch_lightning.utilities.rank_zero: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
[2026-11-14 19:19:51,234] INFO:pytorch_lightning.utilities.rank_zero: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ donor_encoder │ Linear     │ 17.5 M │ train │     0 │
│ 1 │ cell_encoder  │ Sequential │  113 K │ train │     0 │
│ 2 │ attention     │ Sequential │  8.3 K │ train │     0 │
│ 3 │ classifier    │ Sequential │    257 │ train │     0 │
└───┴───────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 17.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 17.6 M                                                                                               
Total estimated model params size (MB): 70.519                                                                     
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Output()

site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


[2026-11-14 19:19:59,827] INFO:pytorch_lightning.utilities.rank_zero: `Trainer.fit` stopped: `max_epochs=2` reached.
